In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from tensorflow.keras.datasets import cifar10
import numpy as np

In [ ]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0   # normalize to [0,1]

INPUT_SHAPE = (32, 32, 3)   # CIFAR-10 images
NUM_CLASSES = 10

In [ ]:
def lenet5():
    """LeNet-5 — LeCun et al. 1998 (adapted for 32x32 color input)"""
    return models.Sequential([
        Input(shape=INPUT_SHAPE),
        layers.Conv2D(6, 5, activation='tanh', padding='same'),
        layers.AveragePooling2D(2),
        layers.Conv2D(16, 5, activation='tanh'),
        layers.AveragePooling2D(2),
        layers.Flatten(),
        layers.Dense(120, activation='tanh'),
        layers.Dense(84, activation='tanh'),
        layers.Dense(NUM_CLASSES, activation='softmax'),
    ], name='LeNet-5')


def alexnet():
    """AlexNet — Krizhevsky et al. 2012 (scaled down for 32x32)"""
    return models.Sequential([
        Input(shape=INPUT_SHAPE),
        layers.Conv2D(96, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(2),
        layers.Conv2D(256, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(2),
        layers.Conv2D(384, 3, activation='relu', padding='same'),
        layers.Conv2D(384, 3, activation='relu', padding='same'),
        layers.Conv2D(256, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(4096, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(4096, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation='softmax'),
    ], name='AlexNet')


def vggnet():
    """VGGNet (VGG-11) — Simonyan & Zisserman 2014 (scaled down for 32x32)"""
    return models.Sequential([
        Input(shape=INPUT_SHAPE),
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(2),
        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(2),
        layers.Conv2D(256, 3, activation='relu', padding='same'),
        layers.Conv2D(256, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(2),
        layers.Conv2D(512, 3, activation='relu', padding='same'),
        layers.Conv2D(512, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation='softmax'),
    ], name='VGGNet')


def residual_block(x, filters):
    """Basic residual block with skip connection"""
    shortcut = x
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    # project shortcut if channel dims differ
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)
    return x


def resnet():
    """ResNet-20 — He et al. 2015 (designed for 32x32, CIFAR variant)"""
    inputs = Input(shape=INPUT_SHAPE)
    x = layers.Conv2D(16, 3, padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    for filters in [16, 32, 64]:
        x = residual_block(x, filters)
        x = residual_block(x, filters)

    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    return models.Model(inputs, outputs, name='ResNet')

In [ ]:
# ─── Train & Compare ─────────────────────────────────────────────────────────

architectures = [lenet5, alexnet, vggnet, resnet]
results = {}

for build_fn in architectures:
    model = build_fn()
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    print(f"\n{'='*50}")
    print(f"  {model.name}")
    print(f"{'='*50}")
    model.summary()

    history = model.fit(
        x_train, y_train,
        epochs=5,
        batch_size=64,
        validation_split=0.1,
        verbose=1
    )

    _, test_acc = model.evaluate(x_test, y_test, verbose=0)
    params = model.count_params()
    results[model.name] = {'test_acc': test_acc, 'params': params}

**Lab Tasks**
Train and compare following CNN models:
LeNet-5
AlexNet
VGGNet
ResNet-20
Visualize Results (Especially Trainable parameters)

Submit ColabNotebook (ipynb) file.